In [ ]:
# 1. Cài đặt thư viện cần thiết
!pip install transformers datasets accelerate evaluate rouge_score -q

import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer
from datasets import load_dataset
from google.colab import drive

# 2. Kết nối Google Drive để lưu Model checkpoint sau khi train
drive.mount('/content/drive')

# 3. Khởi tạo Mô hình & Tokenizer trên GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

# 4. Tải và tiền xử lý tập dữ liệu SAMSum trực tiếp
dataset = load_dataset("knkarthick/samsum")

def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)
    target_encodings = tokenizer(text_target=example_batch['summary'], max_length=128, truncation=True)
    return {
        'input_ids': input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids']
    }

dataset_pt = dataset.map(convert_examples_to_features, batched=True)

# 5. Cấu hình Tham số Huấn luyện (Tối ưu riêng cho GPU T4)
output_dir = "/content/drive/MyDrive/pegasus-samsum-model"
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

trainer_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=2,  # GPU T4 cân tốt batch_size=2 hoặc 4
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16
)

trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_pt["train"],
    eval_dataset=dataset_pt["validation"]
)

# 6. Tiến hành Training & Lưu kết quả
trainer.train()

model_pegasus.save_pretrained(os.path.join(output_dir, "pegasus-samsum-model"))
tokenizer.save_pretrained(os.path.join(output_dir, "tokenizer"))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_path = "/content/drive/MyDrive/pegasus-samsum-model"

# 1. Nạp Tokenizer và Model trực tiếp từ Drive lên GPU
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda")

# 2. Hội thoại mẫu để test
sample_text = """
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check... Sorry, I don't have it.
Hannah: Too bad, I need to contact her asap.
Amanda: Ask Larry, he might have it.
"""

# 3. Tiến hành tóm tắt
inputs = tokenizer(sample_text, return_tensors="pt", max_length=512, truncation=True).to("cuda")
summary_ids = model.generate(inputs["input_ids"], max_length=128, num_beams=4, early_stopping=True)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("--- KẾT QUẢ TÓM TẮT ---")
print(summary)